# Embed Indicator Files

Interactive companion to `embed_indicators.py build` — pick which `.pine`
files in `indicator-scripts/` to embed with Ollama, instead of rebuilding
the whole database.

**What it does**

- lists every `.pine` file with its status in the selected model's DB
  (`new` / `changed` / `stale` / `current` / `in DB but file missing`)
- embeds the selected files exactly like `build`: strip `//` comments ->
  chunk -> Ollama embed -> mean-pool -> store vector + metadata + sha256
- writes the per-model DB (`indicator_embeddings.json` for 0.6b,
  `indicator_embeddings_<model>.json` for other models)

**Cautions**

- each model has its own DB — scores across models are **not comparable**
- the frozen `reference_mean` is **not** recomputed by default (same as
  `build`). Tick *refreeze* to recompute it — that changes **every**
  similarity score, and the thresholds were calibrated on the original
  26-script corpus
- `qwen3-embedding:4b` is ~5-6x slower than `0.6b` (~28 min for the whole
  corpus); a handful of files is fine, the full set is not

**Run order:** top to bottom, then use the panel in the last cell.

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path.cwd()))
import embed_indicators as ei  # reuse strip_comments / chunk_source / embed fns

MODELS_AVAILABLE = ['qwen3-embedding:0.6b', 'qwen3-embedding:4b']

def db_path_for(model):
    """Same per-model DB layout as the CLI (default model -> plain name)."""
    if model == 'qwen3-embedding:0.6b':
        return Path('indicator_embeddings.json')
    return ei.default_db_path(model)

def load_db_for(model):
    path = db_path_for(model)
    if path.exists():
        db = json.loads(path.read_text())
        if db.get('model') != model:
            print(f'{path.name} was built with {db.get("model")!r}, not {model!r}')
            return None
        return db
    return {'model': model, 'indicators': {}}

def file_status(path, db):
    """Classify one .pine file against a loaded DB."""
    entry = db.get('indicators', {}).get(path.name)
    if entry is None or not entry.get('vector'):
        return 'new'
    if entry.get('preprocess') != ei.PREPROCESS_VERSION:
        return f'stale (v{entry.get("preprocess")} -> v{ei.PREPROCESS_VERSION})'
    digest = ei.source_hash(path.read_text(encoding='utf-8', errors='replace'))
    if entry.get('sha256') != digest:
        return 'changed'
    return 'current'

def show_status(model):
    db = load_db_for(model)
    if db is None:
        return
    files = sorted(ei.INDICATOR_DIR.glob('*.pine'))
    print(f'{model}  ->  {db_path_for(model)}  ({len(db["indicators"])} entries)')
    print('-' * 70)
    for f in files:
        print(f'  {file_status(f, db):<40} {f.name}')
    for name in sorted(set(db['indicators']) - {f.name for f in files}):
        print(f'  {"in DB but file missing":<40} {name}')

show_status(MODELS_AVAILABLE[0])

qwen3-embedding:0.6b  ->  indicator_embeddings.json  (26 entries)
----------------------------------------------------------------------
  current                                  00_Zeillerman_Volatility_Trend.pine
  current                                  00_htf_Auction.pine
  current                                  01_CME_Institutional.pine
  current                                  02_Volume_Fib.pine
  current                                  02_zeierman_trend.pine
  current                                  02_zeierman_trend_strategy.pine
  current                                  03_anchored_vwap.pine
  current                                  04_Pulse_mean_acc.pine
  current                                  04_Pulse_mean_acc_strategy.pine
  current                                  05_Monte_Carlo.pine
  current                                  06_Probalistic_forecaster.pine
  current                                  07_Nexus_Fusion.pine
  current                                 

In [2]:
def embed_files(filenames, model, force=False, refreeze=False):
    """Embed the given .pine filenames into the model's DB.

    Same pipeline as `embed_indicators.py build` (hash-skip, metadata,
    reference_mean policy), but only for the selected files — failures on
    one file don't abort the rest, and the DB is still saved.
    """
    ei.MODEL = model
    ei.DB_PATH = db_path_for(model).resolve()
    try:
        db = ei.load_db()
    except SystemExit as e:  # model mismatch in existing DB
        print(e)
        return

    done, skipped, failed = [], [], []
    for name in filenames:
        f = ei.INDICATOR_DIR / name
        if not f.exists():
            print(f'  !!    {name} — file not found, skipped')
            failed.append(name)
            continue
        source = f.read_text(encoding='utf-8', errors='replace')
        digest = ei.source_hash(source)
        entry = db['indicators'].get(name, {})
        if (not force and entry.get('sha256') == digest and entry.get('vector')
                and entry.get('preprocess') == ei.PREPROCESS_VERSION):
            print(f'  skip  {name} (unchanged)')
            skipped.append(name)
            continue
        twins = [n for n, e in db['indicators'].items()
                 if n != name and e.get('sha256') == digest]
        if twins:
            print(f'  note  {name} is byte-identical to {", ".join(twins)}')
        try:
            vec, n_chunks = ei.embed_indicator(source)
        except SystemExit as e:  # Ollama down / model not pulled
            print(f'  FAIL  {name} — {e}')
            failed.append(name)
            continue
        meta = ei.parse_metadata(source)
        now = datetime.now(timezone.utc).isoformat()
        db['indicators'][name] = {
            'sha256': digest,
            'title': meta.get('title', f.stem),
            'author': meta.get('author', meta.get('description')),
            'reward_address': meta.get('reward_address'),
            'reward_bps': meta.get('reward_bps'),
            'lines': source.count('\n') + 1,
            'chunks': n_chunks,
            'dim': int(vec.size),
            'preprocess': ei.PREPROCESS_VERSION,
            'first_inscribed_at': entry.get('first_inscribed_at', now),
            'embedded_at': now,
            'vector': vec.tolist(),
        }
        print(f'  embed {name} ({meta.get("title", f.stem)!r}, '
              f'{n_chunks} chunks, dim={vec.size})', flush=True)
        done.append(name)

    # reference_mean: identical policy to cmd_build — keep frozen unless
    # explicitly asked, missing, or the preprocess version changed.
    stale_mean = db.get('reference_mean_preprocess') != ei.PREPROCESS_VERSION
    if db['indicators'] and (refreeze or stale_mean or 'reference_mean' not in db):
        vecs = np.asarray([e['vector'] for e in db['indicators'].values()])
        db['reference_mean'] = vecs.mean(axis=0).tolist()
        db['reference_mean_n'] = len(vecs)
        db['reference_mean_preprocess'] = ei.PREPROCESS_VERSION
        db['reference_mean_at'] = datetime.now(timezone.utc).isoformat()
        print(f'Refroze reference mean over {len(vecs)} indicators '
              f'(|mean|={np.linalg.norm(vecs.mean(axis=0)):.3f}) — ALL scores shifted')
    else:
        print('reference_mean kept frozen (tick refreeze to recompute)')

    db['preprocess'] = ei.PREPROCESS_VERSION
    ei.save_db(db)
    print(f'Wrote {ei.DB_PATH.name}: {len(done)} embedded, {len(skipped)} skipped, '
          f'{len(failed)} failed — {len(db["indicators"])} indicators total')

    # context preview: each new file's nearest neighbours (centered cosine)
    if done and len(db['indicators']) > 1 and 'reference_mean' in db:
        mu = np.asarray(db['reference_mean'])
        vecs = {n: ei.center(np.asarray(e['vector']), mu)
                for n, e in db['indicators'].items()}
        print('\nNearest neighbours of newly embedded files:')
        for name in done:
            sims = sorted(((ei.cosine(vecs[name], v), n)
                           for n, v in vecs.items() if n != name), reverse=True)[:3]
            print(f'  {name}')
            for s, n in sims:
                tag = ('REMIX?' if s >= ei.REMIX_THRESHOLD
                       else 'related' if s >= ei.RELATED_THRESHOLD else '')
                print(f'    {s:>7.4f}  {tag:<8} {n}')

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    files_all = sorted(f.name for f in ei.INDICATOR_DIR.glob('*.pine'))

    model_dd = widgets.Dropdown(
        options=MODELS_AVAILABLE, value=MODELS_AVAILABLE[0], description='model:',
        layout=widgets.Layout(width='380px'), style={'description_width': '55px'})
    file_sel = widgets.SelectMultiple(
        options=files_all, value=(), description='files:',
        rows=min(16, max(6, len(files_all))),
        layout=widgets.Layout(width='470px'), style={'description_width': '45px'})
    sel_all = widgets.Button(description='Select all', layout=widgets.Layout(width='110px'))
    sel_new = widgets.Button(description='New/changed', layout=widgets.Layout(width='120px'))
    sel_none = widgets.Button(description='Clear', layout=widgets.Layout(width='80px'))
    force_cb = widgets.Checkbox(value=False, description='force re-embed unchanged files')
    refreeze_cb = widgets.Checkbox(
        value=False, description='refreeze reference_mean (changes ALL scores)')
    run_btn = widgets.Button(description='Embed selected', button_style='primary',
                             icon='bolt', layout=widgets.Layout(width='150px'))
    status_out = widgets.Output()
    out = widgets.Output()

    def refresh_status(*_):
        with status_out:
            status_out.clear_output()
            show_status(model_dd.value)

    def _select_new(*_):
        db = load_db_for(model_dd.value) or {'indicators': {}}
        file_sel.value = tuple(f.name for f in sorted(ei.INDICATOR_DIR.glob('*.pine'))
                               if file_status(f, db) != 'current')

    def _run(*_):
        with out:
            out.clear_output()
            if not file_sel.value:
                print('No files selected — pick at least one on the left.')
                return
            run_btn.disabled = True
            try:
                embed_files(list(file_sel.value), model_dd.value,
                            force=force_cb.value, refreeze=refreeze_cb.value)
            finally:
                run_btn.disabled = False
        refresh_status()

    sel_all.on_click(lambda *_: setattr(file_sel, 'value', tuple(files_all)))
    sel_new.on_click(_select_new)
    sel_none.on_click(lambda *_: setattr(file_sel, 'value', ()))
    model_dd.observe(refresh_status, 'value')
    run_btn.on_click(_run)

    refresh_status()
    display(widgets.VBox([
        model_dd,
        status_out,
        widgets.HBox([sel_all, sel_new, sel_none]),
        file_sel,
        force_cb,
        refreeze_cb,
        run_btn,
        out,
    ]))
except ImportError:
    print('ipywidgets not available — call embed_files() directly, e.g.:')
    print('  embed_files(["04_Pulse_mean_acc.pine"], "qwen3-embedding:0.6b")')